In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from dotenv import load_dotenv


# Load API key
load_dotenv()

# Initialize Groq model
model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.7,
)

# Generate response
response = model.invoke("Why do parrots talk?")

print(response.content)

**Short answer:**  
Parrots don’t “talk” the way humans do. They mimic sounds they hear because of a combination of biology (a special vocal organ called the syrinx), brain wiring that lets them learn and reproduce sounds, and social instincts that reward them for making noises that fit in with a group.

---

## 1.  The biology that makes it possible

| Feature | What it does | Why it matters for “talking” |
|---------|--------------|------------------------------|
| **Syrinx** | The bird’s vocal organ, located at the base of the trachea. | It can produce a wide range of pitches, tones, and rhythms—far more flexible than a human larynx. |
| **Neural circuitry** | Parrots have a brain region called the *nucleus accumbens*‑like vocal‑learning pathway, analogous to the human Broca’s area. | This pathway lets them *learn* sounds by listening and then *reproduce* them. |
| **Auditory memory** | They can store and recall complex acoustic patterns. | Allows them to remember a word or phrase l

In [4]:
from langchain.tools import tool

@tool
def get_weather(loaction:str)->str:
    """Get the weather at a loaction"""
    return f"It's sunny in {loaction}"

model_with_tool = model.bind_tools([get_weather])

In [5]:
response = model_with_tool.invoke("What's the weather like in Malout?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to use the get_weather function. Provide location "Malout". The function signature has a typo: "loaction". We must pass that key.', 'tool_calls': [{'id': 'fc_a046792a-58d6-42f1-93d0-5473a6ec9a27', 'function': {'arguments': '{"loaction":"Malout"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 130, 'total_tokens': 188, 'completion_time': 0.064272857, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.006317853, 'prompt_tokens_details': None, 'queue_time': 0.390404397, 'total_time': 0.07059071}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_f102a3179c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a12704-4076-7461-acca-0d8e7f309a9d-0' tool_calls=[{'name': 'get_weather', 'args': {'loaction': 'Malout'}, 'id': 'fc_a046792a-58d6-42f1-93d0-5473a6ec9a

## Tool Execution Loops

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What is the weather in Malout"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)


# Step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 : Pass result back to the model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)

It’s sunny in Malout.


In [7]:
messages

[{'role': 'user', 'content': 'What is the weather in Malout'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in Malout. We have a function get_weather that takes parameter "loaction" (misspelled). We need to call the function with loaction: "Malout". Then we respond with the result.', 'tool_calls': [{'id': 'fc_35debfec-e815-4a3f-8cb7-60a95f30fd4b', 'function': {'arguments': '{"loaction":"Malout"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 129, 'total_tokens': 203, 'completion_time': 0.07457639, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.007160375, 'prompt_tokens_details': None, 'queue_time': 0.044802215, 'total_time': 0.081736765}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1270c-9e4